# PC vs lncRNA — Extended Dataset & Feature-Count Description

Reproduces the **feature processing-step counts** from `000_dataset_descriptions.ipynb`
(raw → numerical → non-constant → continuous/categorical — the numbers behind the paper's
*Table S5*), extended with the revision-only feature groups that now live in the combined
supplementary feature table:

- **ML / Rep / NBD**: original sets, loaded exactly as in `000_dataset_descriptions.ipynb`.
- **DNA repeats** (`dna_*`): RepeatMasker on the unspliced genomic locus — *new*, parallels the
  existing RNA-level Rep set (`rna_*`).
- **rG4 peaks** (`rg4_*`, `has_peak`): rG4detector predictions — *new*.
- **ScanFold**: RNA secondary-structure thermodynamics (avgMFE/avgZ/avgED, paired/competition,
  positional bins) — *new*.

The combined table also carries `rna_*` (Rep) and non-B DNA (`apr_`/`dr_`/`gq_`/.../`total_nonb_`)
columns that duplicate the existing Rep/NBD sets 1:1 — those are loaded from the original pipeline
outputs below rather than double-counted from the combined file.

**Data sources**
- `results/gencode.v47.common.cdhit.cv/tables/gencode.v47.common.cdhit.cv_full_table.tsv` (ML)
- `te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv` (Rep)
- `nonb-pipeline/results/gencode.v47.transcripts/extended_analysis/features_nonb_features.csv` (NBD)
- `revision-wt/results/gencode.v47.common.cdhit.cv/features/supplementary_features.tsv` (DNA repeats, rG4, ScanFold)

## 1. Imports and setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATASET = "gencode.v47.common.cdhit.cv"
BASE = Path("/mnt/cbib/LNClassifier/paper")

sys.path.insert(0, str(BASE / "workflow"))
from utils.features import filter_feature_columns

FULL_TABLE    = BASE / "results" / DATASET / "tables" / f"{DATASET}_full_table.tsv"
TE_FEATURES   = BASE / "te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv"
NBD_FEATURES  = BASE / "nonb-pipeline/results/gencode.v47.transcripts/extended_analysis/features_nonb_features.csv"
SUPP_FEATURES = BASE / "revision-wt/results" / DATASET / "features/supplementary_features.tsv"

TABLE_OUT = BASE / "revision-wt/results" / DATASET / "supplementary_tables" / "element_statistics_tables"
TABLE_OUT.mkdir(parents=True, exist_ok=True)

print("Full table    :", FULL_TABLE)
print("TE features   :", TE_FEATURES)
print("NBD features  :", NBD_FEATURES)
print("Supp features :", SUPP_FEATURES)
print("Table output  :", TABLE_OUT)

Full table    : /mnt/cbib/LNClassifier/paper/results/gencode.v47.common.cdhit.cv/tables/gencode.v47.common.cdhit.cv_full_table.tsv
TE features   : /mnt/cbib/LNClassifier/paper/te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv
NBD features  : /mnt/cbib/LNClassifier/paper/nonb-pipeline/results/gencode.v47.transcripts/extended_analysis/features_nonb_features.csv
Supp features : /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/features/supplementary_features.tsv
Table output  : /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/supplementary_tables/element_statistics_tables


## 2. Load existing sets (ML / Rep / NBD) — unchanged

In [ ]:
# "Raw tables" = total input columns of each source table as delivered by its pipeline,
# including metadata (ids, tool labels, transcript_length/type, ...) -- captured from the
# untouched header before any feature/metadata split.
ml_raw_n  = pd.read_csv(FULL_TABLE, sep="\t", nrows=0).shape[1]
te_raw_n  = pd.read_csv(TE_FEATURES, nrows=0).shape[1]
nbd_raw_n = pd.read_csv(NBD_FEATURES, nrows=0).shape[1]

full_table = pd.read_csv(FULL_TABLE, sep="\t").set_index("seq_ID")
ml_cols = filter_feature_columns(full_table)
ml_df = full_table[ml_cols]

# TE/NBD pipelines are computed genome-wide; restrict rows to the common-CDHIT set analyzed
# here so non-constant/categorical counts are computed over the same transcripts as ML/DNA/rG4/ScanFold.
te_exclude_cols = ["transcript_id", "transcript_length", "transcript_type", "coding_class"]
te_raw = pd.read_csv(TE_FEATURES).set_index("transcript_id").reindex(ml_df.index)
rep_df = te_raw[[c for c in te_raw.columns if c not in te_exclude_cols]]
# The raw TE table stores its "*_has_*" flags as True/False/NaN -> object dtype, which would
# otherwise be dropped entirely at the numeric-selection step below instead of counted as categorical.
obj_cols = rep_df.select_dtypes(include="object").columns
rep_df[obj_cols] = rep_df[obj_cols].apply(pd.to_numeric, errors="coerce")

nbd_raw = pd.read_csv(NBD_FEATURES).set_index("transcript_id").reindex(ml_df.index)
# Rename to avoid this count column being mistaken for a boolean "_present" flag below.
nbd_raw.rename(columns={"motif_types_present": "n_motif_types"}, inplace=True)
nbd_exclude_cols = te_exclude_cols + [c for c in nbd_raw.columns if c.endswith("_transcript_id")]
nbd_df = nbd_raw[[c for c in nbd_raw.columns if c not in nbd_exclude_cols]]

print(f"ML  : {ml_raw_n} raw input columns -> {ml_df.shape[1]} feature columns, {ml_df.shape[0]} transcripts")
print(f"Rep : {te_raw_n} raw input columns -> {rep_df.shape[1]} feature columns, {rep_df.shape[0]} transcripts")
print(f"NBD : {nbd_raw_n} raw input columns -> {nbd_df.shape[1]} feature columns, {nbd_df.shape[0]} transcripts")

Identified length columns to exclude: ['Transcript_length_lncDC', 'length_plncpro'] (keeping RNA_size_feelnc for reference)
Total number of columns in features table: 172
Number of kept feature columns: 128
Feature columns: ['kmerScore_1mer_feelnc', 'kmerScore_2mer_feelnc', 'kmerScore_3mer_feelnc', 'kmerScore_6mer_feelnc', 'kmerScore_9mer_feelnc', 'kmerScore_12mer_feelnc', 'ORF_cover_feelnc', 'RNA_size_feelnc', 'ORF_l_cpat', 'Fickett_l_cpat', 'Hexamer_l_cpat', 'ORF_coverage_l_cpat', 'GC_content_lncDC', 'Fickett_score_lncDC', 'ORF_T0_length_lncDC', 'ORF_T1_length_lncDC', 'ORF_T2_length_lncDC', 'ORF_T0_coverage_lncDC', 'ORF_T1_coverage_lncDC', 'ORF_T3_coverage_lncDC', 'Hexamer_score_ORF_T0_lncDC', 'Hexamer_score_ORF_T1_lncDC', 'Hexamer_score_ORF_T2_lncDC', 'Hexamer_score_ORF_T3_lncDC', 'RCB_T0_lncDC', 'RCB_T1_lncDC', 'ORF_T0_PI_lncDC', 'ORF_T0_MW_lncDC', 'ORF_T0_aromaticity_lncDC', 'ORF_T0_instability_lncDC', 'ORF_T1_MW_lncDC', 'ORF_T1_instability_lncDC', 'ORF_T2_MW_lncDC', 'ORF_T3_MW_ln

## 3. Load novel pipelines (DNA repeats / rG4 / ScanFold)

Feature groups are split by column-name prefix from the combined table; only the groups absent
from the pipelines loaded above are pulled in (`usecols` keeps the read light — the file is ~0.5 GB).

In [ ]:
supp_cols = pd.read_csv(SUPP_FEATURES, sep="\t", nrows=0).columns

NONB_PREFIXES = ("apr_", "dr_", "gq_plus_", "gq_minus_", "ir_", "mr_", "str_", "tri_", "z_",
                  "all_nonb_", "any_nonb_", "motif_", "total_nonb_")

dna_cols = [c for c in supp_cols if c.startswith("dna_")]
rg4_cols = [c for c in supp_cols if c.startswith("rg4_") or c == "has_peak"]
scanfold_cols = [c for c in supp_cols
                 if c not in ("seq_ID", "coding_class", "unspliced_length")
                 and not c.startswith(("rna_", "dna_", "rg4_"))
                 and c != "has_peak"
                 and not c.startswith(NONB_PREFIXES)]

novel_cols = dna_cols + rg4_cols + scanfold_cols
supp_df = pd.read_csv(SUPP_FEATURES, sep="\t", usecols=["seq_ID"] + novel_cols).set_index("seq_ID")

# Cast bool -> int before reindexing: a NaN-introducing reindex would otherwise upcast
# bool columns to object and drop them from the numeric-dtype counts below.
bool_cols = supp_df.select_dtypes(include="bool").columns
supp_df[bool_cols] = supp_df[bool_cols].astype(int)

missing = ml_df.index.difference(supp_df.index)
if len(missing):
    print(f"Note: {len(missing)} ML transcripts have no supplementary_features.tsv row "
          f"(see supplementary_excluded.tsv) — reindexed as NaN.")
supp_df = supp_df.reindex(ml_df.index)  # align rows to the common-CDHIT set used above

dna_df = supp_df[dna_cols]
rg4_df = supp_df[rg4_cols]
scanfold_df = supp_df[scanfold_cols]

print(f"DNA repeats : {dna_df.shape[1]} raw feature columns, {dna_df.shape[0]} transcripts")
print(f"rG4 peaks   : {rg4_df.shape[1]} raw feature columns")
print(f"ScanFold    : {scanfold_df.shape[1]} raw feature columns")

Note: 612 ML transcripts have no supplementary_features.tsv row (see supplementary_excluded.tsv) — reindexed as NaN.
DNA repeats : 170 raw feature columns, 112264 transcripts
rG4 peaks   : 4 raw feature columns
ScanFold    : 63 raw feature columns


## 4. Feature-count breakdown per processing step

Same reduction as `000_dataset_descriptions.ipynb`: numerical dtype only → drop constant columns
→ split what remains into continuous vs. categorical (`_has_` / `_present` substrings, plus any
explicit exceptions).

In [ ]:
def feature_step_counts(df, name, raw_n=None, cat_substrings=("_has_", "_present"), extra_categorical=()):
    """Raw -> numerical -> non-constant -> continuous/categorical counts for one feature set.

    raw_n: total input columns of the source table, including metadata (ids, labels, ...).
    Defaults to df.shape[1] for sets with no separate raw table (DNA repeats/rG4/ScanFold,
    which are column slices of the shared combined feature table, not standalone pipelines).
    """
    if raw_n is None:
        raw_n = df.shape[1]

    numeric_df = df.select_dtypes(include=[np.number])

    nunique = numeric_df.nunique()
    non_constant_cols = nunique[nunique > 1].index.tolist()
    non_constant_df = numeric_df[non_constant_cols]

    cat_cols = [c for c in non_constant_df.columns
                if any(s in c for s in cat_substrings) or c in extra_categorical]
    cont_cols = [c for c in non_constant_df.columns if c not in cat_cols]

    counts = {
        "Set": name,
        "Raw tables": raw_n,
        "Numerical features": numeric_df.shape[1],
        "Non-constant": non_constant_df.shape[1],
        "Continuous": len(cont_cols),
        "Categorical": len(cat_cols),
    }
    print(f"{name}: {counts}")
    return counts

In [ ]:
rows = [
    feature_step_counts(ml_df,       "ML",  raw_n=ml_raw_n),
    feature_step_counts(rep_df,      "Rep", raw_n=te_raw_n),
    feature_step_counts(nbd_df,      "NBD", raw_n=nbd_raw_n),
    feature_step_counts(dna_df,      "DNA repeats"),
    feature_step_counts(rg4_df,      "rG4", extra_categorical=("has_peak",)),
    feature_step_counts(scanfold_df, "ScanFold"),
]
step_counts = pd.DataFrame(rows).set_index("Set")
step_counts.loc["Total"] = step_counts.sum()
step_counts = step_counts.T  # steps as rows, sets as columns -- matches Table S5 layout
step_counts

ML: {'Set': 'ML', 'Raw tables': 173, 'Numerical features': 128, 'Non-constant': 128, 'Continuous': 128, 'Categorical': 0}
Rep: {'Set': 'Rep', 'Raw tables': 174, 'Numerical features': 170, 'Non-constant': 157, 'Continuous': 142, 'Categorical': 15}
NBD: {'Set': 'NBD', 'Raw tables': 212, 'Numerical features': 199, 'Non-constant': 199, 'Continuous': 189, 'Categorical': 10}
DNA repeats: {'Set': 'DNA repeats', 'Raw tables': 170, 'Numerical features': 170, 'Non-constant': 157, 'Continuous': 142, 'Categorical': 15}
rG4: {'Set': 'rG4', 'Raw tables': 4, 'Numerical features': 4, 'Non-constant': 4, 'Continuous': 3, 'Categorical': 1}
ScanFold: {'Set': 'ScanFold', 'Raw tables': 63, 'Numerical features': 63, 'Non-constant': 63, 'Continuous': 63, 'Categorical': 0}


Set,ML,Rep,NBD,DNA repeats,rG4,ScanFold,Total
Raw tables,173,174,212,170,4,63,796
Numerical features,128,170,199,170,4,63,734
Non-constant,128,157,199,157,4,63,708
Continuous,128,142,189,142,3,63,667
Categorical,0,15,10,15,1,0,41


## 4.1 Per group feature inspection



In [ ]:
te_feats  = pd.read_csv(TE_FEATURES)

In [ ]:
te_feats.head()

,transcript_id,transcript_length,te_min_sw_score,te_max_sw_score,te_mean_sw_score,te_min_divergence,te_max_divergence,te_mean_divergence,te_min_perc_del,te_max_perc_del,...,lctr_simple_repeat_count_per_kb,lctr_satellite_count_per_kb,pseudo_count_per_kb,pseudo_rrna_count_per_kb,pseudo_scrna_count_per_kb,pseudo_snrna_count_per_kb,pseudo_trna_count_per_kb,pseudo_ancient_count_per_kb,transcript_type,coding_class
0,ENST00000832824.1,1379,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.725163,0.0,0.0,0.0,0.0,0.0,0.0,lncRNA,lncRNA
1,ENST00000832825.1,1417,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.705716,0.0,0.0,0.0,0.0,0.0,0.0,lncRNA,lncRNA
2,ENST00000832826.1,1538,509.0,509.0,509.0,19.9,19.9,19.9,18.2,18.2,...,0.0,0.650195,0.0,0.0,0.0,0.0,0.0,0.0,lncRNA,lncRNA
3,ENST00000832827.1,1519,509.0,509.0,509.0,19.9,19.9,19.9,18.2,18.2,...,0.0,0.658328,0.0,0.0,0.0,0.0,0.0,0.0,lncRNA,lncRNA
4,ENST00000832828.1,1762,509.0,509.0,509.0,19.9,19.9,19.9,18.2,18.2,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,lncRNA,lncRNA


In [ ]:
te_feats.columns.str.split("_").str[0].value_counts()

te            81
pseudo        42
lctr          21
unknown       12
global        10
pseudogene     4
transcript     3
coding         1
Name: count, dtype: int64

## 5. Export

In [ ]:
out_path = TABLE_OUT / "S5_feature_counts_extended.csv"
step_counts.to_csv(out_path, index_label="Step")
print("Wrote:", out_path, step_counts.shape)

Wrote: /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/supplementary_tables/element_statistics_tables/S5_feature_counts_extended.csv (5, 7)
